# OrientDB tutorial

## Prerequisites

### Documentation

You will find all documentation for :
* [OrientDB SQL reference](http://www.orientdb.com/docs/last/SQL-Functions.html)
* [Orientdb python client](http://orientdb.com/docs/last/PyOrient-Client.html#working-with-the-client)

## Import libraries

In [1]:
import pyorient


In [2]:
ROOT_PASSWORD = "root"
client = pyorient.OrientDB("localhost", 2424)
session_id = client.connect("root", ROOT_PASSWORD)

In [3]:
print(client.db_list())

{{'databases': {}}}


## I. Quick start

### Creating the database

**Q:** Create a database `gods` as a `GRAPH_DATABASE` in `MEMORY_STORAGE_TYPE`. 

We will use it to store relationships between Greek deities.

In [4]:
DB_name = "gods"
client.db_create(
        DB_name, 
        pyorient.DB_TYPE_GRAPH, 
        pyorient.STORAGE_TYPE_MEMORY
    )

**Q:** Connect your pyorient client to the `gods` database.

In [5]:
client.db_open("gods", "root", ROOT_PASSWORD)

**Q:** You should now be able to launch OrientDB queries through the Python client with the [command()](http://orientdb.com/docs/last/PyOrient-Client-Command.html) function. 

You should think of OrientDB as a Graph-Document database for the following questions. Each vertex and edge will contain information on it inside a JSON document.

Create a new Vertex with content `{name: 'Zeus', symbol: 'thunder'}`. The [CREATE VERTEX : Create a vertex using JSON content](http://orientdb.com/docs/last/SQL-Create-Vertex.html) doc page should help you.

In [6]:
#creation de la classe God
client.command("CREATE CLASS God EXTENDS V")


# Insertion de Zeus
sql = "CREATE VERTEX God SET name = 'Zeus', symbol = 'thunder'"
result = client.command(sql)

You have created a VERTEX in the previous question. The VERTEX is a [class](https://orientdb.com/docs/last/Tutorial-Classes.html) of OrientDB which defines a record that can be linked to others through EDGE instances.

You can find all VERTEX created in the database with a SQL command on the `V` table, like `SELECT * FROM V`. 

**Q:** Print all current vertices in `gods`, it should only have `Zeus` though for now.

In [7]:
vertices = client.command("SELECT * FROM V")
for v in vertices:
   
    data = v.oRecordData
    rid = v._rid
    print(f"ID: {rid} | Nom: {data.get('name')} | Symbole: {data.get('symbol')}")

ID: #17:0 | Nom: Zeus | Symbole: thunder


**Q:** Create new vertices with content : 
```
{name:Héra, symbol:tiara}
{name:Poséidon, symbol:trident}
{name:Athena, symbol:helmet}
{name:Arès, symbol:weapons} 
```

In [8]:
new_gods = [
    {'name': 'Héra', 'symbol': 'tiara'},
    {'name': 'Poséidon', 'symbol': 'trident'},
    {'name': 'Athena', 'symbol': 'helmet'},
    {'name': 'Arès', 'symbol': 'weapons'}
]

for god in new_gods:
    sql = f"CREATE VERTEX God SET name = '{god['name']}', symbol = '{god['symbol']}'"
    
    result = client.command(sql)
    rid = result[0]._rid


**Q:** Display all vertices with name = `Arès`

In [9]:
sql_query = f"SELECT FROM V WHERE name = 'Arès'"

results = client.command(sql_query)
for vertex in results:
        data = vertex.oRecordData
        rid = vertex._rid
        print(f"RID: {rid} | Nom: {data.get('name')} | Symbole: {data.get('symbol')}")

RID: #17:1 | Nom: Arès | Symbole: weapons


**Q:** Create an EDGE from `Zeus` to `Poséidon` with the content `{kind: 'sibling'}

In [19]:
sql_edge = f"""
    CREATE EDGE E 
    FROM (SELECT FROM God WHERE name = 'Zeus') 
    TO (SELECT FROM God WHERE name = 'Poséidon')
    SET kind = 'sibling'
"""
edge_result = client.command(sql_edge)



**Q:** Redisplay all vertices, discuss.

In [15]:
vertices = client.command("SELECT * FROM V")
for v in vertices:
   
    data = v.oRecordData
    rid = v._rid
    print(f"ID: {rid} | Nom: {data.get('name')} | Symbole: {data.get('symbol')}")

ID: #17:0 | Nom: Zeus | Symbole: thunder
ID: #17:1 | Nom: Arès | Symbole: weapons
ID: #18:0 | Nom: Héra | Symbole: tiara
ID: #19:0 | Nom: Poséidon | Symbole: trident
ID: #20:0 | Nom: Athena | Symbole: helmet


**Q:** Display all edges. They are contained in the class `E`

In [21]:
edges = client.command("SELECT * FROM E")
for e in edges:
   
    source_rid = e.out
    target_rid = getattr(e, 'in')
    relation_type = e.oRecordData.get('kind', 'N/A')
    print(f" Edge ID: {e._rid}; Direction : {source_rid} --({relation_type})--> {target_rid}")

 Edge ID: #13:0; Direction : #17:0 --(sibling)--> #19:0
 Edge ID: #14:0; Direction : #17:0 --(sibling)--> #19:0


Two fields on vertices have appeared, containing the outgoing (out_) and incoming (in_) links.

At the edge level, two fields point to the original (out) and destination (in) vertices.

**Q:** Lets create some more edges :

* Zeus > Héra (sibling)
* Zeus > Arès (father)
* Zeus > Athena (father)
* Héra > Arès (mother)
* Héra > Zeus (sibling)
* Poséidon > Zeus (sibling)

_Hint 1 :_ check [the CREATE EDGE doc page](http://orientdb.com/docs/last/SQL-Create-Edge.html) to find an example for creating edges on vertices using subqueries so you can run queries to fetch the vertices before creating an edge in between.

_Hint 2 :_ after you have found the command to create edges between vertices with sub-queries, you should be well-versed enough in Python to create a list of all edges in the question, and loop the command on each element of the list to create all edges in one go =)

In [22]:
relations = [
    {'from': 'Zeus', 'to': 'Héra', 'kind': 'sibling'},
    {'from': 'Zeus', 'to': 'Arès', 'kind': 'father'},
    {'from': 'Zeus', 'to': 'Athena', 'kind': 'father'},
    {'from': 'Héra', 'to': 'Arès', 'kind': 'mother'},
    {'from': 'Héra', 'to': 'Zeus', 'kind': 'sibling'},
    {'from': 'Poséidon', 'to': 'Zeus', 'kind': 'sibling'}
]

for rel in relations:
    
    sql_edge = f"""
        CREATE EDGE E 
        FROM (SELECT FROM God WHERE name = '{rel['from']}') 
        TO (SELECT FROM God WHERE name = '{rel['to']}')
        SET kind = '{rel['kind']}'
    """
    client.command(sql_edge)
    print(f"Lien créé : {rel['from']} --({rel['kind']})--> {rel['to']}")

Lien créé : Zeus --(sibling)--> Héra
Lien créé : Zeus --(father)--> Arès
Lien créé : Zeus --(father)--> Athena
Lien créé : Héra --(mother)--> Arès
Lien créé : Héra --(sibling)--> Zeus
Lien créé : Poséidon --(sibling)--> Zeus


### Looking for data

**Q:** Using [out()](http://orientdb.com/docs/last/Tutorial-Working-with-graphs.html#querying-graphs) function, display all vertices connected and outgoing from Zeus.

You should use the EXPAND() special function to transform the vertex collection in the result-set by expanding it, making the results more readable.

In [23]:
sql_zeus_connections = "SELECT EXPAND(out()) FROM God WHERE name = 'Zeus'"

connections = client.command(sql_zeus_connections)
for node in connections:
        # On accède aux données du sommet (Héra, Arès, Athena, etc.)
        name = node.oRecordData.get('name', 'Inconnu')
        rid = node._rid
        
        print(f"Dieu connecté : {name} (ID: {rid})")

Dieu connecté : Poséidon (ID: #19:0)
Dieu connecté : Poséidon (ID: #19:0)
Dieu connecté : Héra (ID: #18:0)
Dieu connecté : Arès (ID: #17:1)
Dieu connecté : Athena (ID: #20:0)


**Q:** Display all vertices which got a father (the vertices which are the destination of an arc whose kind attribute is father).

_Hint: You can notice that we use the field `in` the arc, and not the function `in()` which applies to vertices._

In [25]:
sql_fathers_dest = "SELECT EXPAND(in) FROM E WHERE kind = 'father'"
children = client.command(sql_fathers_dest)
children_rids = set()
    
for child in children:
    rid = child._rid
    if rid not in children_rids:
        name = child.oRecordData.get('name', 'Inconnu')
        print(f"Enfant : {name} (ID: {rid})")
        children_rids.add(rid)

Enfant : Athena (ID: #20:0)
Enfant : Arès (ID: #17:1)


**Q:** As in SQL, the operator `in` used in a clause `where` allows to restrict the possible values with an embedded query _(where ... in (select ...))_. 

Display the mothers, by displaying the vertices where an outgoing arc is part of the arcs where kind is a mother.

In [26]:
sql_mothers = "SELECT EXPAND(out) FROM E WHERE kind = 'mother'"

mothers_result = client.command(sql_mothers)
mothers_rids = set()

for mother in mothers_result:
    rid = mother._rid
    if rid not in mothers_rids:
        name = mother.oRecordData.get('name', 'Inconnu')
        print(f"Mère identifiée : {name} (ID: {rid})")
        mothers_rids.add(rid)

Mère identifiée : Héra (ID: #18:0)


**Q:** Display the brothers and sisters of Zeus (the destination summits of an arc whose kind is sibling and whose original summit is Zeus).

In [33]:
sql_siblings = """
    SELECT EXPAND( bothE()[kind='sibling'].bothV() ) 
    FROM God 
    WHERE name = 'Zeus'
    
"""
results = client.command(sql_siblings)
siblings_rids = set()
    
for sibling in results:
    rid=sibling._rid
    if rid not in siblings_rids :
        name = sibling.oRecordData.get('name', 'Inconnu')
        print(f"Fratrie : {name} (ID: {rid})")
        siblings_rids.add(rid)

Fratrie : Héra (ID: #18:0)
Fratrie : Zeus (ID: #17:0)
Fratrie : Poséidon (ID: #19:0)


## Modeling a Product Recommendation System

You are currently modeling the data of a product recommendation system with OrientDB.

The main purpose of such a system is to answer the question "which products were purchased by their people who purchased product X? »

Purchased products have only one name field. They are purchased by people who have a nickname.

When a person buys a product, the date of purchase is stored. 

Instead of working with "anonymous" vertices and arcs, you will use classes. The `create class` command allows you to create custom classes.

The vertex classes must extend V, the arc classes must extend E.

**Q:** Create an `eCommerce` database, and the necessary classes to model the system.

PS : you can view all classes in the database with :

```python
for name in client.command("SELECT name FROM (SELECT expand(classes) FROM metadata:schema)"):
    print(name)
```

In [34]:
DB_NAME = "eCommerce"
if client.db_exists(DB_NAME):
    client.db_drop(DB_NAME)

client.db_create(DB_NAME, pyorient.DB_TYPE_GRAPH, pyorient.STORAGE_TYPE_MEMORY)
client.db_open(DB_NAME, "root", ROOT_PASSWORD)

# Classe pour les personnes 
client.command("CREATE CLASS Person EXTENDS V")
client.command("CREATE PROPERTY Person.name STRING")

# Classe pour les produits 
client.command("CREATE CLASS Product EXTENDS V")
client.command("CREATE PROPERTY Product.name STRING")

# Classe pour l'acte d'achat (Arc)

client.command("CREATE CLASS Purchased EXTENDS E")
client.command("CREATE PROPERTY Purchased.date DATETIME")

[1]

**Q:** Create the following products: `spaghetti`, `bolognese sauce`, `cheese`, `apple`.

In [35]:
products = ["spaghetti", "bolognese sauce", "cheese", "apple"]
for p_name in products:
    sql = f"CREATE VERTEX Product SET name = '{p_name}'"
    
    client.command(sql)
    print(f"Produit ajouté : {p_name}")
    


Produit ajouté : spaghetti
Produit ajouté : bolognese sauce
Produit ajouté : cheese
Produit ajouté : apple


**Q:** Create the following people: `peter`, `meredith`.

In [37]:
clients=["peter","meredith"]

for c_name in clients :
    sql= f"CREATE VERTEX Person SET name='{c_name}'"
    client.command(sql)
    print(f"Client ajouté: {c_name}")

Client ajouté: peter
Client ajouté: meredith


**Q:** Create the following purchases: 
- peter > spaghetti + cheese on 20/01/2016 
- meredith > cheese + apple + bolognese sauce on 22/01/2016
- peter > spaghetti + bolognese sauce on 27/01/2016


In [39]:
purchases = [
    ("peter", ["spaghetti", "cheese"], "2016-01-20"),
    ("meredith", ["cheese", "apple", "bolognese sauce"], "2016-01-22"),
    ("peter", ["spaghetti", "bolognese sauce"], "2016-01-27")
]
for name, product_list, p_date in purchases:
    for product_name in product_list:
        
        sql_purchase = f"""
            CREATE EDGE Purchased 
            FROM (SELECT FROM Person WHERE name = '{name}') 
            TO (SELECT FROM Product WHERE name = '{product_name}') 
            SET date = '{p_date}'
        """
        client.command(sql_purchase)
        print(f"Achat enregistré : {name} -> {product_name} le {p_date}")
        

Achat enregistré : peter -> spaghetti le 2016-01-20
Achat enregistré : peter -> cheese le 2016-01-20
Achat enregistré : meredith -> cheese le 2016-01-22
Achat enregistré : meredith -> apple le 2016-01-22
Achat enregistré : meredith -> bolognese sauce le 2016-01-22
Achat enregistré : peter -> spaghetti le 2016-01-27
Achat enregistré : peter -> bolognese sauce le 2016-01-27


**Q:** Who bought Bolognese sauce?

In [40]:
sql_clients = f"""
    SELECT EXPAND(in('Purchased')) 
    FROM Product 
    WHERE name = 'bolognese sauce'
"""
buyers = client.command(sql_clients)
buyers_found = set()

for person in buyers:
            name = person.oRecordData.get('name')
            if name not in buyers_found:
                print(f"Client : {name}")
                buyers_found.add(name)

Client : meredith
Client : peter


**Q:** It is possible to link the `out` and `in` navigation functions. What products are purchased with Bolognese sauce? 

In [43]:
sql_buy_with = f"""
    SELECT FROM (
        SELECT EXPAND(in('Purchased').out('Purchased')) 
        FROM Product 
        WHERE name = 'bolognese sauce'
    ) WHERE name <> 'bolognese sauce'
"""
buy_with = client.command(sql_buy_with)
unique_names = set(prod.name for prod in buy_with)
for name in unique_names:
    print(f"- {name}")

- apple
- spaghetti
- cheese


## Postquisites

Since we create databases in memory, they get destroyed on server shutdown.